# META-CXR Training on Kaggle (2x T4 GPU)

This notebook trains the META-CXR model on the MIMIC-CXR-JPG dataset using 2x T4 GPUs via PyTorch DistributedDataParallel.

**Prerequisites:**
- Kaggle accelerator set to **GPU T4 x2**
- `GCS_SERVICE_ACCOUNT` secret added in Kaggle notebook settings (JSON key for the GCS service account with read access to `gs://mimic-cxr-jpg-lite`)
- Internet access enabled

**Steps:** Run cells 1→7 in order.

## Cell 1 — Install Dependencies

In [ ]:
import subprocess, sys

# Kaggle has PyTorch, torchvision, numpy, pandas, scikit-learn preinstalled.
# Install only the packages that are missing.
packages = [
    "omegaconf==2.3.0",
    "pycocoevalcap",
    "scikit-image",           # latest stable; io/transform APIs are unchanged
    "torchinfo",
    "wandb",
    "loralib==0.1.1",
    "iterative-stratification",
    "iopath",
    "timm==0.6.13",           # 0.6.x keeps timm.models.hub API used by dist_utils.py; PyTorch 2.x compatible
    "spacy",                  # latest 3.x; stable spacy.load() API
    "nltk==3.8.1",
    "google-cloud-storage",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + packages,
    check=True
)

# Install peft at the specific commit used by the project
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "git+https://github.com/huggingface/peft.git@e536616888d51b453ed354a6f1e243fecb02ea08"],
    check=True
)

# Download NLTK data required by the METEOR scorer
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Download spacy English model required by blip2.py (spacy.load("en_core_web_sm"))
subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)

# Detect Java installation (needed for METEOR/ROUGE scoring)
result = subprocess.run(
    "readlink -f $(which java) | sed 's|/bin/java||'",
    shell=True, capture_output=True, text=True
)
JAVA_HOME_DETECTED = result.stdout.strip()
print(f"Detected JAVA_HOME: {JAVA_HOME_DETECTED}")

# Verify GPU count
import torch
print(f"GPUs available: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

## Cell 2 — GCS Authentication

Add your GCS service account JSON as a Kaggle Secret named `GCS_SERVICE_ACCOUNT`:
- Notebook → Add-ons → Secrets → Add new secret
- Name: `GCS_SERVICE_ACCOUNT`
- Value: paste the full JSON content of your service account key file

In [ ]:
import os
import json
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
raw = user_secrets.get_secret("GCS_SERVICE_ACCOUNT").strip()

parsed = json.loads(raw)
if isinstance(parsed, str):
    parsed = json.loads(parsed)

credentials_path = "/kaggle/working/gcs_credentials.json"
with open(credentials_path, "w") as f:
    json.dump(parsed, f)

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = credentials_path
print("GCS credentials written to:", credentials_path)

from google.cloud import storage
client = storage.Client(project="mimic-cxr-jpg-491409")
bucket = client.bucket("mimic-cxr-jpg-lite")
print(f"Bucket 'mimic-cxr-jpg-lite' accessible: {bucket.exists()}")

## Cell 3 — Clone GitHub Repository

In [ ]:
import os

REPO_DIR = "/kaggle/working/META-CXR"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/minhphuong150505/Meta-CXR-Kaggle.git {REPO_DIR}
else:
    print(f"Repository already exists at {REPO_DIR}, pulling latest changes...")
    !git -C {REPO_DIR} pull

# Change working directory to repo root
os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
!ls -la

## Cell 4 — Download MIMIC-CXR Data from GCS

Expected bucket structure in `gs://mimic-cxr-jpg-lite/`:
```
mimic-cxr-jpg/2.1.0/
    mimic-cxr-2.0.0-split.csv
    files/p10.../  (JPG images)
mimic-cxr/report_processed/
    mimic_cxr_cleaned.csv
data/data_files/
    mimic-cxr-2.0.0-chexpert.csv
    mimic-cxr-2.0.0-metadata.csv
```

In [ ]:
import os

GCS_BUCKET = "gs://mimic-cxr-jpg-lite"
DATA_ROOT = "/kaggle/working/data"
os.makedirs(DATA_ROOT, exist_ok=True)

# Inspect bucket structure first
print("=== Bucket top-level structure ===")
!gsutil ls {GCS_BUCKET}/

print("\n=== Syncing data from GCS (this may take a while for images) ===")
# Use -m for parallel downloads; rsync only copies missing/changed files on reruns
!gsutil -m rsync -r {GCS_BUCKET}/ {DATA_ROOT}/

# Verify the 4 critical CSV files are present
print("\n=== Verifying critical files ===")
critical_files = [
    f"{DATA_ROOT}/mimic-cxr-jpg/2.1.0/mimic-cxr-2.0.0-split.csv",
    f"{DATA_ROOT}/mimic-cxr/report_processed/mimic_cxr_cleaned.csv",
    f"{DATA_ROOT}/data/data_files/mimic-cxr-2.0.0-chexpert.csv",
    f"{DATA_ROOT}/data/data_files/mimic-cxr-2.0.0-metadata.csv",
]
all_ok = True
for f in critical_files:
    exists = os.path.exists(f)
    status = "OK " if exists else "MISSING"
    print(f"  [{status}] {f}")
    if not exists:
        all_ok = False

if all_ok:
    print("\nAll critical files found. Ready to train.")
else:
    print("\nSome files are missing. Check the bucket structure and adjust DATA_ROOT paths.")

## Cell 5 — Write `configs/env_config.yaml` with Kaggle Paths

In [ ]:
import os
import subprocess

# Dynamically detect Java home on Kaggle
result = subprocess.run(
    "readlink -f $(which java) | sed 's|/bin/java||'",
    shell=True, capture_output=True, text=True
)
java_home = result.stdout.strip() or "/usr/lib/jvm/java-8-openjdk-amd64/jre"
java_path = java_home + "/bin:"

DATA_ROOT = "/kaggle/working/data"

env_config_content = f"""paths:
  data_root: "{DATA_ROOT}"
  mimic_cxr_jpg_root: "${{paths.data_root}}/mimic-cxr-jpg/2.1.0"
  split_csv: "${{paths.mimic_cxr_jpg_root}}/mimic-cxr-2.0.0-split.csv"
  reports_csv: "${{paths.data_root}}/mimic-cxr/report_processed/mimic_cxr_cleaned.csv"
  chexpert_csv: "${{paths.data_root}}/data/data_files/mimic-cxr-2.0.0-chexpert.csv"
  metadata_csv: "${{paths.data_root}}/data/data_files/mimic-cxr-2.0.0-metadata.csv"
  output_dir: "/kaggle/working/output"
  checkpoint_dir: "/kaggle/working/checkpoints"
  gcs_bucket: "gs://mimic-cxr-jpg-lite"
  gcs_project: "mimic-cxr-jpg-491409"

wandb:
  entity: ""
  project: "meta-cxr"

java:
  home: "{java_home}"
  path: "{java_path}"
"""

os.makedirs("configs", exist_ok=True)
with open("configs/env_config.yaml", "w") as f:
    f.write(env_config_content)

print("Written configs/env_config.yaml:")
print(env_config_content)

## Cell 6 — Launch 2-GPU DDP Training

Uses `torch.distributed.run` (alias for `torchrun`) with `--standalone` for single-node multi-GPU.  
Training output is streamed live. Expect each epoch to take 30–90 minutes depending on dataset size.

In [ ]:
import subprocess
import sys
import os

os.makedirs("/kaggle/working/output", exist_ok=True)
os.makedirs("/kaggle/working/checkpoints", exist_ok=True)

cmd = [
    sys.executable, "-m", "torch.distributed.run",
    "--standalone",
    "--nproc_per_node=2",
    "--master_port=12355",
    "-m", "pretraining.train",
    "--cfg-path", "pretraining/configs/mimic_cxr_2gpu.yaml",
]

print("Launch command:")
print(" ".join(cmd))
print("\n" + "="*60 + "\n")

env = os.environ.copy()
env["PYTHONPATH"] = "/kaggle/working/META-CXR"

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd="/kaggle/working/META-CXR",
    env=env,
)

for line in process.stdout:
    print(line, end="", flush=True)

process.wait()
print(f"\n" + "="*60)
print(f"Training finished with exit code: {process.returncode}")

## Cell 7 — Display Evaluation Results

In [ ]:
import json
import os
import glob
import pandas as pd

OUTPUT_DIR = "/kaggle/working/output"

# ── Training logs ────────────────────────────────────────────────────────────
log_files = sorted(glob.glob(f"{OUTPUT_DIR}/**/log.txt", recursive=True))
print(f"Found {len(log_files)} log file(s)")

for log_file in log_files:
    print(f"\n{'='*60}")
    print(f"Log: {log_file}")
    print('='*60)
    records = []
    with open(log_file) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                print(line)
    if records:
        df = pd.DataFrame(records)
        display(df)

# ── Prediction files ─────────────────────────────────────────────────────────
pred_files = sorted(glob.glob(f"{OUTPUT_DIR}/**/predictions_*.txt", recursive=True))
print(f"\nFound {len(pred_files)} prediction file(s)")

for pred_file in pred_files[:2]:
    print(f"\n{'='*60}")
    print(f"Predictions: {pred_file}")
    print('='*60)
    with open(pred_file) as f:
        for i, line in enumerate(f):
            if i >= 10:
                print(f"  ... ({sum(1 for _ in open(pred_file))} total lines)")
                break
            print(line, end="")

# ── Checkpoint summary ───────────────────────────────────────────────────────
checkpoints = sorted(glob.glob(f"{OUTPUT_DIR}/**/checkpoint_*.pth", recursive=True))
print(f"\nSaved checkpoints ({len(checkpoints)}):")
for ckpt in checkpoints:
    size_mb = os.path.getsize(ckpt) / (1024 ** 2)
    print(f"  {ckpt}  ({size_mb:.1f} MB)")

# ── Best checkpoint info ─────────────────────────────────────────────────────
best_ckpts = glob.glob(f"{OUTPUT_DIR}/**/checkpoint_best.pth", recursive=True)
if best_ckpts:
    print(f"\nBest checkpoint: {best_ckpts[0]}")